In [1]:
import pandas as pd 
import numpy as np
import requests

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2020"

tables = pd.read_html(url, header=0, storage_options={"User-Agent": "Mozilla/5.0"})

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]

In [3]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,3,The Grudge,Screen Gems / Stage 6 Films / Ghost House Pict...,Nicolas Pesce (director/screenplay); Andrea Ri...,[2]
1,J A N U A R Y,10,Underwater,20th Century Fox / Chernin Entertainment,"William Eubank (director); Brian Duffield, Ada...",[3]
2,J A N U A R Y,10,Like a Boss,Paramount Pictures / Artists First,"Miguel Arteta (director); Sam Pitman, Adam Col...",[4]
3,J A N U A R Y,10,Three Christs,IFC Films,Jon Avnet (director/screenplay); Eric Nazarian...,NaN
4,J A N U A R Y,10,Inherit the Viper,Lionsgate / Barry Films / Tycor International ...,Anthony Jerjen (director); Andrew Crabtree (sc...,[5]
...,...,...,...,...,...,...
274,D E C E M B E R,25,We Can Be Heroes,Netflix / Troublemaker Studios / Double R Prod...,Robert Rodriguez (director/screenplay); Priyan...,[247]
275,D E C E M B E R,25,News of the World,Universal Pictures / Playtone / Perfect World ...,Paul Greengrass (director/screenplay); Luke Da...,[248]
276,D E C E M B E R,25,One Night in Miami...,Amazon Studios,Regina King (director); Kemp Powers (screenpla...,[249]
277,D E C E M B E R,25,Promising Young Woman,Focus Features / FilmNation Entertainment,Emerald Fennell (director/screenplay); Carey M...,[250]


In [4]:
from tmdbv3api import TMDb
import json

tmdb = TMDb()
tmdb.api_key = 'bc6771260f54114bb3e5d552c1776130'

In [11]:
from tmdbv3api import Movie

tmdb_movie = Movie()

def get_genres(x):
    result = tmdb_movie.search(x)

    if not result:
        return np.nan

    try:
        movie_id = result[0].id
    except Exception:
        return np.nan

    response = requests.get(
        f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}'
    )
    data_json = response.json()

    if 'genres' in data_json and data_json['genres']:
        return " ".join([g['name'] for g in data_json['genres']])
    
    return np.nan

In [12]:
df['genres'] = df['Title'].map(lambda x: get_genres(str(x)))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres
0,J A N U A R Y,3,The Grudge,Screen Gems / Stage 6 Films / Ghost House Pict...,Nicolas Pesce (director/screenplay); Andrea Ri...,[2],Horror Mystery Thriller
1,J A N U A R Y,10,Underwater,20th Century Fox / Chernin Entertainment,"William Eubank (director); Brian Duffield, Ada...",[3],Horror Science Fiction Action Adventure
2,J A N U A R Y,10,Like a Boss,Paramount Pictures / Artists First,"Miguel Arteta (director); Sam Pitman, Adam Col...",[4],Comedy
3,J A N U A R Y,10,Three Christs,IFC Films,Jon Avnet (director/screenplay); Eric Nazarian...,NaN,Drama
4,J A N U A R Y,10,Inherit the Viper,Lionsgate / Barry Films / Tycor International ...,Anthony Jerjen (director); Andrew Crabtree (sc...,[5],Crime Thriller Drama
...,...,...,...,...,...,...,...
274,D E C E M B E R,25,We Can Be Heroes,Netflix / Troublemaker Studios / Double R Prod...,Robert Rodriguez (director/screenplay); Priyan...,[247],Family Action Fantasy Comedy
275,D E C E M B E R,25,News of the World,Universal Pictures / Playtone / Perfect World ...,Paul Greengrass (director/screenplay); Luke Da...,[248],Drama Western Adventure
276,D E C E M B E R,25,One Night in Miami...,Amazon Studios,Regina King (director); Kemp Powers (screenpla...,[249],Drama
277,D E C E M B E R,25,Promising Young Woman,Focus Features / FilmNation Entertainment,Emerald Fennell (director/screenplay); Carey M...,[250],Thriller Crime Drama


In [13]:
def get_director(x):
    directors = []
    parts = x.split("; ")
    
    for p in parts:
        if "(director)" in p or "(directors)" in p or "(director/screenplay)" in p:
            directors.append(p.split(" (")[0])
    
    if len(directors) == 0:
        return np.nan
    
    return ", ".join(directors)

In [14]:
df['director_name'] = df['Cast and crew'].map(lambda x: get_director(x))

In [16]:
def get_actor1(x):
    parts = x.split("; ")
    
    for p in parts:
        if "(" not in p:
            actors = p.split(", ")
            if len(actors) > 0:
                return actors[0]
    
    return np.nan

In [17]:
df['actor_1_name'] = df['Cast and crew'].map(lambda x: get_actor1(x))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name,actor_1_name
0,J A N U A R Y,3,The Grudge,Screen Gems / Stage 6 Films / Ghost House Pict...,Nicolas Pesce (director/screenplay); Andrea Ri...,[2],Horror Mystery Thriller,Nicolas Pesce,Andrea Riseborough
1,J A N U A R Y,10,Underwater,20th Century Fox / Chernin Entertainment,"William Eubank (director); Brian Duffield, Ada...",[3],Horror Science Fiction Action Adventure,William Eubank,Kristen Stewart
2,J A N U A R Y,10,Like a Boss,Paramount Pictures / Artists First,"Miguel Arteta (director); Sam Pitman, Adam Col...",[4],Comedy,Miguel Arteta,Tiffany Haddish
3,J A N U A R Y,10,Three Christs,IFC Films,Jon Avnet (director/screenplay); Eric Nazarian...,NaN,Drama,Jon Avnet,Richard Gere
4,J A N U A R Y,10,Inherit the Viper,Lionsgate / Barry Films / Tycor International ...,Anthony Jerjen (director); Andrew Crabtree (sc...,[5],Crime Thriller Drama,Anthony Jerjen,Josh Hartnett
...,...,...,...,...,...,...,...,...,...
274,D E C E M B E R,25,We Can Be Heroes,Netflix / Troublemaker Studios / Double R Prod...,Robert Rodriguez (director/screenplay); Priyan...,[247],Family Action Fantasy Comedy,Robert Rodriguez,Priyanka Chopra Jonas
275,D E C E M B E R,25,News of the World,Universal Pictures / Playtone / Perfect World ...,Paul Greengrass (director/screenplay); Luke Da...,[248],Drama Western Adventure,Paul Greengrass,Tom Hanks
276,D E C E M B E R,25,One Night in Miami...,Amazon Studios,Regina King (director); Kemp Powers (screenpla...,[249],Drama,Regina King,Kingsley Ben-Adir
277,D E C E M B E R,25,Promising Young Woman,Focus Features / FilmNation Entertainment,Emerald Fennell (director/screenplay); Carey M...,[250],Thriller Crime Drama,Emerald Fennell,Carey Mulligan


In [18]:
def get_actor2(x):
    parts = x.split("; ")
    
    for p in parts:
        if "(" not in p:
            actors = p.split(", ")
            if len(actors) > 1:
                return actors[1]
    
    return np.nan

In [19]:
df['actor_2_name'] = df['Cast and crew'].map(lambda x: get_actor2(x))

In [22]:
def get_actor3(x):
    parts = x.split("; ")
    
    for p in parts:
        if "(" not in p:
            actors = p.split(", ")
            if len(actors) > 2:
                return actors[2]
    
    return np.nan

In [23]:
df['actor_3_name'] = df['Cast and crew'].map(lambda x: get_actor3(x))

In [24]:
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,J A N U A R Y,3,The Grudge,Screen Gems / Stage 6 Films / Ghost House Pict...,Nicolas Pesce (director/screenplay); Andrea Ri...,[2],Horror Mystery Thriller,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho
1,J A N U A R Y,10,Underwater,20th Century Fox / Chernin Entertainment,"William Eubank (director); Brian Duffield, Ada...",[3],Horror Science Fiction Action Adventure,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick
2,J A N U A R Y,10,Like a Boss,Paramount Pictures / Artists First,"Miguel Arteta (director); Sam Pitman, Adam Col...",[4],Comedy,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek
3,J A N U A R Y,10,Three Christs,IFC Films,Jon Avnet (director/screenplay); Eric Nazarian...,NaN,Drama,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins
4,J A N U A R Y,10,Inherit the Viper,Lionsgate / Barry Films / Tycor International ...,Anthony Jerjen (director); Andrew Crabtree (sc...,[5],Crime Thriller Drama,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs
...,...,...,...,...,...,...,...,...,...,...,...
274,D E C E M B E R,25,We Can Be Heroes,Netflix / Troublemaker Studios / Double R Prod...,Robert Rodriguez (director/screenplay); Priyan...,[247],Family Action Fantasy Comedy,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin
275,D E C E M B E R,25,News of the World,Universal Pictures / Playtone / Perfect World ...,Paul Greengrass (director/screenplay); Luke Da...,[248],Drama Western Adventure,Paul Greengrass,Tom Hanks,Helena Zengel,NaN
276,D E C E M B E R,25,One Night in Miami...,Amazon Studios,Regina King (director); Kemp Powers (screenpla...,[249],Drama,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge
277,D E C E M B E R,25,Promising Young Woman,Focus Features / FilmNation Entertainment,Emerald Fennell (director/screenplay); Carey M...,[250],Thriller Crime Drama,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie


In [25]:
df = df.loc[:, ["director_name", "actor_1_name", "actor_2_name", "actor_3_name", "genres", "Title"]]
df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,Title
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery Thriller,The Grudge
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,Underwater
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,Like a Boss
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,Three Christs
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,Inherit the Viper


In [26]:
df = df.rename(columns={"Title": "movie_title"})
df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery Thriller,The Grudge
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,Underwater
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,Like a Boss
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,Three Christs
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,Inherit the Viper


In [27]:
df.isnull().sum()

director_name     4
actor_1_name      3
actor_2_name      8
actor_3_name     30
genres            3
movie_title       0
dtype: int64

In [28]:
df['director_name'] = df['director_name'].fillna("unknown")
df['actor_1_name'] = df['actor_1_name'].fillna("unknown")
df['actor_2_name'] = df['actor_2_name'].fillna("unknown")
df['actor_3_name'] = df['actor_3_name'].fillna("unknown")
df['genres'] = df['genres'].fillna("unknown")

In [29]:
df.isnull().sum()

director_name    0
actor_1_name     0
actor_2_name     0
actor_3_name     0
genres           0
movie_title      0
dtype: int64

In [30]:
df['comb'] = df['actor_1_name'] + ' ' + df['actor_2_name'] + ' ' + df['actor_3_name'] + ' ' + df['director_name'] + ' ' + df['genres']

In [31]:
df

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery Thriller,The Grudge,Andrea Riseborough Demián Bichir John Cho Nico...
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,Underwater,Kristen Stewart Vincent Cassel Jessica Henwick...
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,Like a Boss,Tiffany Haddish Rose Byrne Salma Hayek Miguel ...
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,Three Christs,Richard Gere Peter Dinklage Walton Goggins Jon...
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,Inherit the Viper,Josh Hartnett Margarita Levieva Chandler Riggs...
...,...,...,...,...,...,...,...
274,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin,Family Action Fantasy Comedy,We Can Be Heroes,Priyanka Chopra Jonas Pedro Pascal YaYa Gossel...
275,Paul Greengrass,Tom Hanks,Helena Zengel,unknown,Drama Western Adventure,News of the World,Tom Hanks Helena Zengel unknown Paul Greengras...
276,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge,Drama,One Night in Miami...,Kingsley Ben-Adir Eli Goree Aldis Hodge Regina...
277,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie,Thriller Crime Drama,Promising Young Woman,Carey Mulligan Bo Burnham Alison Brie Emerald ...


In [32]:
finaldf = pd.read_csv("../Data/processed/final_combined_movie_metadata.csv")
finaldf

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
5861,"Nick Bruno, Troy Quane",Will Smith,Tom Holland,Rashida Jones,Animation Action Adventure Comedy Family,spies in disguise,Will Smith Tom Holland Rashida Jones Nick Brun...
5862,Greta Gerwig,Saoirse Ronan,Emma Watson,Florence Pugh,Drama Romance History,little women,Saoirse Ronan Emma Watson Florence Pugh Greta ...
5863,Sam Mendes,George MacKay,Dean-Charles Chapman,Mark Strong,War History,1917,George MacKay Dean-Charles Chapman Mark Strong...
5864,Destin Daniel Cretton,Michael B. Jordan,Jamie Foxx,Brie Larson,Drama Crime History,just mercy,Michael B. Jordan Jamie Foxx Brie Larson Desti...


In [33]:
final = pd.concat([finaldf, df], ignore_index=True)
final

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
6140,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin,Family Action Fantasy Comedy,We Can Be Heroes,Priyanka Chopra Jonas Pedro Pascal YaYa Gossel...
6141,Paul Greengrass,Tom Hanks,Helena Zengel,unknown,Drama Western Adventure,News of the World,Tom Hanks Helena Zengel unknown Paul Greengras...
6142,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge,Drama,One Night in Miami...,Kingsley Ben-Adir Eli Goree Aldis Hodge Regina...
6143,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie,Thriller Crime Drama,Promising Young Woman,Carey Mulligan Bo Burnham Alison Brie Emerald ...


In [34]:
final.isnull().sum()

director_name    0
actor_1_name     0
actor_2_name     0
actor_3_name     0
genres           0
movie_title      0
comb             0
dtype: int64

In [35]:
final.duplicated().sum()  

np.int64(0)

In [37]:
final.shape

(6145, 7)

In [38]:
final.to_csv("../Data/processed/final_combined_movie_metadata.csv", index=False)